# 🎵 스펙트로그램 패턴 및 파동 분석

이 노트북에서는 Mel Spectrogram을 사용하여 다양한 패턴과 파동 분석을 수행합니다.

## 📋 목차
1. **데이터 로드 및 스펙트로그램 추출**
2. **기본 통계 분석** (max, min, mean, std, median, quartiles)
3. **시간 축 패턴 분석** (각 주파수 밴드의 시간적 변화)
4. **주파수 축 패턴 분석** (각 시간 프레임의 주파수 분포)
5. **에너지 분포 분석**
6. **주파수 밴드별 통계 비교**
7. **시간 구간별 통계 분석**
8. **스펙트로그램 변화율 분석** (gradient)
9. **PCA 시각화 및 군집 분석**



In [ ]:
# ============================================================
# 필수 라이브러리 임포트
# ============================================================

import os
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 머신러닝
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score, adjusted_rand_score, 
    normalized_mutual_info_score
)

# 프로젝트 모듈
from app.ml.features.extractor import AudioFeatureExtractor, AudioConfig

# 시각화 설정
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("✅ 라이브러리 로드 완료!")



## 1. 데이터 로드 및 스펙트로그램 추출



In [ ]:
# ============================================================
# 데이터 로드 및 스펙트로그램 추출
# ============================================================

# 피처 추출기 초기화
config = AudioConfig()
feature_extractor = AudioFeatureExtractor(config)

# 데이터 경로
data_dir = Path('../data')
all_files = []
all_states = []

# 파일 수집
print("📂 데이터 파일 수집 중...")
for state_dir in sorted(data_dir.iterdir()):
    if not state_dir.is_dir() or state_dir.name == 'augmented':
        continue
    
    state_name = state_dir.name
    state_idx = {'braking state': 0, 'idle state': 1, 'startup state': 2}.get(state_name, -1)
    
    if state_idx == -1:
        continue
    
    for problem_dir in state_dir.iterdir():
        if not problem_dir.is_dir() or problem_dir.name == 'combined':
            continue
        
        for file_path in problem_dir.glob('*.wav'):
            all_files.append(file_path)
            all_states.append(state_idx)

print(f"✅ 총 {len(all_files)}개 파일 수집 완료!")

# 스펙트로그램 추출 (2D 배열로 유지)
print("\n🔄 스펙트로그램 추출 중...")
all_spectrograms = []
all_spectrograms_2d = []  # 2D 배열로 저장 (패턴 분석용)

for file_path in tqdm(all_files, desc="스펙트로그램 추출"):
    try:
        # Mel Spectrogram 추출 (2D 배열: frequency × time)
        mel_spec = feature_extractor.extract_mel_spectrogram(
            *feature_extractor.load_audio(str(file_path))
        )
        
        # 2D 배열 저장 (패턴 분석용)
        all_spectrograms_2d.append(mel_spec)
        
        # Flatten: 2D → 1D 벡터로 변환 (PCA/클러스터링용)
        mel_spec_flat = mel_spec.flatten()
        all_spectrograms.append(mel_spec_flat)
        
    except Exception as e:
        print(f"⚠️  오류: {file_path} - {e}")
        # 오류 시 0으로 채운 배열 추가
        default_shape = (128, 216)  # 기본 shape
        all_spectrograms_2d.append(np.zeros(default_shape))
        all_spectrograms.append(np.zeros(128 * 216))

# NumPy 배열로 변환
X_flat = np.array(all_spectrograms)  # (samples, features) - flatten된 버전
X_2d = np.array(all_spectrograms_2d)  # (samples, frequency, time) - 2D 버전
y_states = np.array(all_states)

state_names = ['braking', 'idle', 'startup']

print(f"\n✅ 스펙트로그램 추출 완료!")
print(f"   X_flat shape: {X_flat.shape}")  # 예: (2832, 27648)
print(f"   X_2d shape: {X_2d.shape}")  # 예: (2832, 128, 216)
print(f"   y_states shape: {y_states.shape}")  # 예: (2832,)



## 2. 기본 통계 분석 (max, min, mean, std, median, quartiles)



In [ ]:
# ============================================================
# 기본 통계 분석 (max, min, mean, std, median, quartiles)
# ============================================================

def extract_statistical_features(spectrogram_2d):
    """
    스펙트로그램의 다양한 통계적 특징을 추출
    
    Returns:
        dict: 통계적 특징들
    """
    features = {}
    
    # 전체 스펙트로그램 통계
    features['mean'] = np.mean(spectrogram_2d)
    features['std'] = np.std(spectrogram_2d)
    features['max'] = np.max(spectrogram_2d)
    features['min'] = np.min(spectrogram_2d)
    features['median'] = np.median(spectrogram_2d)
    features['q25'] = np.percentile(spectrogram_2d, 25)
    features['q75'] = np.percentile(spectrogram_2d, 75)
    features['iqr'] = features['q75'] - features['q25']  # Interquartile Range
    features['range'] = features['max'] - features['min']
    features['skewness'] = np.mean(((spectrogram_2d - features['mean']) / features['std']) ** 3) if features['std'] > 0 else 0
    features['kurtosis'] = np.mean(((spectrogram_2d - features['mean']) / features['std']) ** 4) if features['std'] > 0 else 0
    
    # 에너지 관련
    features['energy'] = np.sum(spectrogram_2d ** 2)
    features['rms'] = np.sqrt(np.mean(spectrogram_2d ** 2))  # Root Mean Square
    
    return features

# 모든 샘플에 대해 통계 특징 추출
print("🔄 통계 특징 추출 중...")
statistical_features = []

for spec_2d in tqdm(X_2d, desc="통계 특징 추출"):
    features = extract_statistical_features(spec_2d)
    statistical_features.append(features)

# DataFrame으로 변환
df_stats = pd.DataFrame(statistical_features)

print(f"\n✅ 통계 특징 추출 완료! (Shape: {df_stats.shape})")
print(f"\n📊 추출된 통계 특징 목록:")
print(df_stats.columns.tolist())

# 상태별 통계 요약
print("\n" + "=" * 70)
print("📊 상태별 통계 요약")
print("=" * 70)
for state_idx, state_name in enumerate(state_names):
    mask = y_states == state_idx
    print(f"\n🔵 {state_name.upper()}:")
    print(df_stats[mask].describe().round(4))



In [ ]:
# ============================================================
# 상태별 통계 특징 비교 시각화
# ============================================================

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

# 주요 통계 특징들
key_features = ['mean', 'std', 'max', 'min', 'median', 'q25', 'q75', 'iqr', 
                'range', 'energy', 'rms', 'skewness']

for idx, feature_name in enumerate(key_features):
    ax = axes[idx]
    
    # 상태별 박스플롯
    data_to_plot = [df_stats[y_states == i][feature_name].values for i in range(3)]
    
    bp = ax.boxplot(data_to_plot, labels=state_names, patch_artist=True)
    
    # 박스 색상 설정
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(f'{feature_name}', fontweight='bold')
    ax.set_ylabel('값')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('📊 상태별 통계 특징 비교', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# 시간 축 패턴 분석 (각 주파수 밴드의 시간적 변화)
# ============================================================

def extract_time_pattern_features(spectrogram_2d):
    """
    각 주파수 밴드의 시간적 변화 패턴을 분석
    
    Returns:
        dict: 시간 축 패턴 특징들
    """
    features = {}
    
    # 각 주파수 밴드별 시간 평균 및 표준편차
    freq_band_means = np.mean(spectrogram_2d, axis=1)  # (frequency,)
    freq_band_stds = np.std(spectrogram_2d, axis=1)    # (frequency,)
    
    # 주파수 밴드별 통계
    features['freq_band_mean_mean'] = np.mean(freq_band_means)
    features['freq_band_mean_std'] = np.std(freq_band_means)
    features['freq_band_std_mean'] = np.mean(freq_band_stds)
    features['freq_band_std_std'] = np.std(freq_band_stds)
    
    # 시간 축 변화율 (gradient)
    time_gradient = np.gradient(spectrogram_2d, axis=1)  # 시간 축으로 미분
    features['time_gradient_mean'] = np.mean(time_gradient)
    features['time_gradient_std'] = np.std(time_gradient)
    features['time_gradient_max'] = np.max(time_gradient)
    features['time_gradient_min'] = np.min(time_gradient)
    
    # 시간 축 변화율의 변화율 (가속도)
    time_gradient2 = np.gradient(time_gradient, axis=1)
    features['time_gradient2_mean'] = np.mean(time_gradient2)
    features['time_gradient2_std'] = np.std(time_gradient2)
    
    # 저주파/중주파/고주파 영역대별 시간 패턴
    n_freq = spectrogram_2d.shape[0]
    low_band = spectrogram_2d[:n_freq//3, :]
    mid_band = spectrogram_2d[n_freq//3:2*n_freq//3, :]
    high_band = spectrogram_2d[2*n_freq//3:, :]
    
    for band_name, band_data in [('low', low_band), ('mid', mid_band), ('high', high_band)]:
        band_time_mean = np.mean(band_data, axis=0)  # 각 시간 프레임의 평균
        features[f'{band_name}_time_mean'] = np.mean(band_time_mean)
        features[f'{band_name}_time_std'] = np.std(band_time_mean)
        features[f'{band_name}_time_max'] = np.max(band_time_mean)
        features[f'{band_name}_time_min'] = np.min(band_time_mean)
    
    return features

# 모든 샘플에 대해 시간 패턴 특징 추출
print("🔄 시간 패턴 특징 추출 중...")
time_pattern_features = []

for spec_2d in tqdm(X_2d, desc="시간 패턴 추출"):
    features = extract_time_pattern_features(spec_2d)
    time_pattern_features.append(features)

# DataFrame으로 변환
df_time_pattern = pd.DataFrame(time_pattern_features)

print(f"\n✅ 시간 패턴 특징 추출 완료! (Shape: {df_time_pattern.shape})")



In [ ]:
# ============================================================
# 시간 패턴 시각화 (샘플 예시)
# ============================================================

# 각 상태별로 대표 샘플 선택
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

for state_idx, state_name in enumerate(state_names):
    # 해당 상태의 첫 번째 샘플 선택
    state_mask = y_states == state_idx
    sample_idx = np.where(state_mask)[0][0]
    sample_spec = X_2d[sample_idx]
    
    # 1. 전체 스펙트로그램
    im1 = axes[state_idx, 0].imshow(sample_spec, aspect='auto', origin='lower', cmap='viridis')
    axes[state_idx, 0].set_title(f'{state_name} - 전체 스펙트로그램', fontweight='bold')
    axes[state_idx, 0].set_xlabel('Time')
    axes[state_idx, 0].set_ylabel('Frequency')
    plt.colorbar(im1, ax=axes[state_idx, 0])
    
    # 2. 저주파/중주파/고주파 영역대별 시간 평균
    n_freq = sample_spec.shape[0]
    low_mean = np.mean(sample_spec[:n_freq//3, :], axis=0)
    mid_mean = np.mean(sample_spec[n_freq//3:2*n_freq//3, :], axis=0)
    high_mean = np.mean(sample_spec[2*n_freq//3:, :], axis=0)
    
    axes[state_idx, 1].plot(low_mean, label='Low', color='#FF6B6B', alpha=0.7)
    axes[state_idx, 1].plot(mid_mean, label='Mid', color='#4ECDC4', alpha=0.7)
    axes[state_idx, 1].plot(high_mean, label='High', color='#45B7D1', alpha=0.7)
    axes[state_idx, 1].set_title(f'{state_name} - 주파수 영역대별 시간 평균', fontweight='bold')
    axes[state_idx, 1].set_xlabel('Time')
    axes[state_idx, 1].set_ylabel('Amplitude')
    axes[state_idx, 1].legend()
    axes[state_idx, 1].grid(True, alpha=0.3)
    
    # 3. 주파수 밴드별 시간 표준편차
    freq_band_stds = np.std(sample_spec, axis=1)
    axes[state_idx, 2].plot(freq_band_stds, color='#9B59B6', linewidth=2)
    axes[state_idx, 2].set_title(f'{state_name} - 주파수 밴드별 시간 표준편차', fontweight='bold')
    axes[state_idx, 2].set_xlabel('Frequency Band')
    axes[state_idx, 2].set_ylabel('Std')
    axes[state_idx, 2].grid(True, alpha=0.3)

plt.suptitle('🎵 상태별 시간 패턴 분석', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()



## 4. 주파수 축 패턴 분석 (각 시간 프레임의 주파수 분포)



In [ ]:
# ============================================================
# 주파수 축 패턴 분석 (각 시간 프레임의 주파수 분포)
# ============================================================

def extract_frequency_pattern_features(spectrogram_2d):
    """
    각 시간 프레임의 주파수 분포 패턴을 분석
    
    Returns:
        dict: 주파수 축 패턴 특징들
    """
    features = {}
    
    # 각 시간 프레임별 주파수 평균 및 표준편차
    time_frame_means = np.mean(spectrogram_2d, axis=0)  # (time,)
    time_frame_stds = np.std(spectrogram_2d, axis=0)    # (time,)
    
    # 시간 프레임별 통계
    features['time_frame_mean_mean'] = np.mean(time_frame_means)
    features['time_frame_mean_std'] = np.std(time_frame_means)
    features['time_frame_std_mean'] = np.mean(time_frame_stds)
    features['time_frame_std_std'] = np.std(time_frame_stds)
    
    # 주파수 축 변화율 (gradient)
    freq_gradient = np.gradient(spectrogram_2d, axis=0)  # 주파수 축으로 미분
    features['freq_gradient_mean'] = np.mean(freq_gradient)
    features['freq_gradient_std'] = np.std(freq_gradient)
    features['freq_gradient_max'] = np.max(freq_gradient)
    features['freq_gradient_min'] = np.min(freq_gradient)
    
    # 주파수 축 변화율의 변화율 (가속도)
    freq_gradient2 = np.gradient(freq_gradient, axis=0)
    features['freq_gradient2_mean'] = np.mean(freq_gradient2)
    features['freq_gradient2_std'] = np.std(freq_gradient2)
    
    # 주파수 중심 (Spectral Centroid 유사)
    freq_indices = np.arange(spectrogram_2d.shape[0])
    weighted_freq = np.sum(freq_indices[:, np.newaxis] * spectrogram_2d, axis=0) / (np.sum(spectrogram_2d, axis=0) + 1e-10)
    features['spectral_centroid_mean'] = np.mean(weighted_freq)
    features['spectral_centroid_std'] = np.std(weighted_freq)
    
    # 주파수 대역별 에너지 비율
    n_freq = spectrogram_2d.shape[0]
    low_energy = np.sum(spectrogram_2d[:n_freq//3, :] ** 2)
    mid_energy = np.sum(spectrogram_2d[n_freq//3:2*n_freq//3, :] ** 2)
    high_energy = np.sum(spectrogram_2d[2*n_freq//3:, :] ** 2)
    total_energy = low_energy + mid_energy + high_energy + 1e-10
    
    features['low_energy_ratio'] = low_energy / total_energy
    features['mid_energy_ratio'] = mid_energy / total_energy
    features['high_energy_ratio'] = high_energy / total_energy
    
    return features

# 모든 샘플에 대해 주파수 패턴 특징 추출
print("🔄 주파수 패턴 특징 추출 중...")
freq_pattern_features = []

for spec_2d in tqdm(X_2d, desc="주파수 패턴 추출"):
    features = extract_frequency_pattern_features(spec_2d)
    freq_pattern_features.append(features)

# DataFrame으로 변환
df_freq_pattern = pd.DataFrame(freq_pattern_features)

print(f"\n✅ 주파수 패턴 특징 추출 완료! (Shape: {df_freq_pattern.shape})")



In [ ]:
# ============================================================
# 주파수 패턴 시각화 (샘플 예시)
# ============================================================

# 각 상태별로 대표 샘플 선택
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

for state_idx, state_name in enumerate(state_names):
    # 해당 상태의 첫 번째 샘플 선택
    state_mask = y_states == state_idx
    sample_idx = np.where(state_mask)[0][0]
    sample_spec = X_2d[sample_idx]
    
    # 1. 전체 스펙트로그램
    im1 = axes[state_idx, 0].imshow(sample_spec, aspect='auto', origin='lower', cmap='viridis')
    axes[state_idx, 0].set_title(f'{state_name} - 전체 스펙트로그램', fontweight='bold')
    axes[state_idx, 0].set_xlabel('Time')
    axes[state_idx, 0].set_ylabel('Frequency')
    plt.colorbar(im1, ax=axes[state_idx, 0])
    
    # 2. 각 시간 프레임별 주파수 평균
    time_frame_means = np.mean(sample_spec, axis=0)
    axes[state_idx, 1].plot(time_frame_means, color='#E74C3C', linewidth=2)
    axes[state_idx, 1].set_title(f'{state_name} - 시간 프레임별 주파수 평균', fontweight='bold')
    axes[state_idx, 1].set_xlabel('Time Frame')
    axes[state_idx, 1].set_ylabel('Mean Amplitude')
    axes[state_idx, 1].grid(True, alpha=0.3)
    
    # 3. 주파수 밴드별 평균 (전체 시간에 대한 평균)
    freq_band_means = np.mean(sample_spec, axis=1)
    axes[state_idx, 2].plot(freq_band_means, color='#3498DB', linewidth=2)
    axes[state_idx, 2].set_title(f'{state_name} - 주파수 밴드별 평균', fontweight='bold')
    axes[state_idx, 2].set_xlabel('Frequency Band')
    axes[state_idx, 2].set_ylabel('Mean Amplitude')
    axes[state_idx, 2].grid(True, alpha=0.3)

plt.suptitle('🎵 상태별 주파수 패턴 분석', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()



## 5. 에너지 분포 분석



In [ ]:
# ============================================================
# 에너지 분포 분석
# ============================================================

# 상태별 에너지 분포 비교
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 전체 에너지 분포
energy_values = df_stats['energy'].values
for state_idx, state_name in enumerate(state_names):
    mask = y_states == state_idx
    axes[0, 0].hist(energy_values[mask], bins=50, alpha=0.6, 
                    label=state_name, color=['#FF6B6B', '#4ECDC4', '#45B7D1'][state_idx])
axes[0, 0].set_xlabel('Energy')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('전체 에너지 분포', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. RMS 분포
rms_values = df_stats['rms'].values
for state_idx, state_name in enumerate(state_names):
    mask = y_states == state_idx
    axes[0, 1].hist(rms_values[mask], bins=50, alpha=0.6, 
                    label=state_name, color=['#FF6B6B', '#4ECDC4', '#45B7D1'][state_idx])
axes[0, 1].set_xlabel('RMS')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('RMS 분포', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. 주파수 영역대별 에너지 비율 비교
low_energy_ratios = df_freq_pattern['low_energy_ratio'].values
mid_energy_ratios = df_freq_pattern['mid_energy_ratio'].values
high_energy_ratios = df_freq_pattern['high_energy_ratio'].values

x_pos = np.arange(len(state_names))
width = 0.25

low_means = [np.mean(low_energy_ratios[y_states == i]) for i in range(3)]
mid_means = [np.mean(mid_energy_ratios[y_states == i]) for i in range(3)]
high_means = [np.mean(high_energy_ratios[y_states == i]) for i in range(3)]

axes[1, 0].bar(x_pos - width, low_means, width, label='Low', color='#FF6B6B', alpha=0.7)
axes[1, 0].bar(x_pos, mid_means, width, label='Mid', color='#4ECDC4', alpha=0.7)
axes[1, 0].bar(x_pos + width, high_means, width, label='High', color='#45B7D1', alpha=0.7)
axes[1, 0].set_xlabel('State')
axes[1, 0].set_ylabel('Energy Ratio')
axes[1, 0].set_title('주파수 영역대별 에너지 비율', fontweight='bold')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(state_names)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. 에너지 vs RMS 산점도
for state_idx, state_name in enumerate(state_names):
    mask = y_states == state_idx
    axes[1, 1].scatter(energy_values[mask], rms_values[mask], 
                      alpha=0.5, label=state_name, 
                      color=['#FF6B6B', '#4ECDC4', '#45B7D1'][state_idx], s=20)
axes[1, 1].set_xlabel('Energy')
axes[1, 1].set_ylabel('RMS')
axes[1, 1].set_title('Energy vs RMS', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('⚡ 에너지 분포 분석', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()



## 6. 주파수 밴드별 통계 비교



In [ ]:
# ============================================================
# 주파수 밴드별 통계 비교
# ============================================================

# 모든 샘플의 주파수 밴드별 평균 계산
print("🔄 주파수 밴드별 통계 계산 중...")

# 각 상태별로 주파수 밴드별 평균 계산
freq_band_stats = {state_name: [] for state_name in state_names}

for state_idx, state_name in enumerate(state_names):
    state_mask = y_states == state_idx
    state_specs = X_2d[state_mask]
    
    # 각 주파수 밴드별 평균 계산
    freq_band_means = np.mean(state_specs, axis=(0, 2))  # (frequency,)
    freq_band_stds = np.std(state_specs, axis=(0, 2))    # (frequency,)
    
    freq_band_stats[state_name] = {
        'mean': freq_band_means,
        'std': freq_band_stds
    }

# 시각화
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# 1. 주파수 밴드별 평균
for state_idx, state_name in enumerate(state_names):
    axes[0].plot(freq_band_stats[state_name]['mean'], 
                label=state_name, linewidth=2,
                color=['#FF6B6B', '#4ECDC4', '#45B7D1'][state_idx])
axes[0].set_xlabel('Frequency Band')
axes[0].set_ylabel('Mean Amplitude')
axes[0].set_title('주파수 밴드별 평균 비교', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. 주파수 밴드별 표준편차
for state_idx, state_name in enumerate(state_names):
    axes[1].plot(freq_band_stats[state_name]['std'], 
                label=state_name, linewidth=2,
                color=['#FF6B6B', '#4ECDC4', '#45B7D1'][state_idx])
axes[1].set_xlabel('Frequency Band')
axes[1].set_ylabel('Std Amplitude')
axes[1].set_title('주파수 밴드별 표준편차 비교', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('📊 주파수 밴드별 통계 비교', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("✅ 주파수 밴드별 통계 비교 완료!")



## 7. 시간 구간별 통계 분석



In [ ]:
# ============================================================
# 시간 구간별 통계 분석
# ============================================================

def extract_time_segment_features(spectrogram_2d, n_segments=3):
    """
    시간을 여러 구간으로 나누어 각 구간의 통계를 분석
    
    Args:
        spectrogram_2d: (frequency, time) 스펙트로그램
        n_segments: 시간 구간 개수
    
    Returns:
        dict: 시간 구간별 특징들
    """
    features = {}
    n_time = spectrogram_2d.shape[1]
    segment_size = n_time // n_segments
    
    for seg_idx in range(n_segments):
        start = seg_idx * segment_size
        end = (seg_idx + 1) * segment_size if seg_idx < n_segments - 1 else n_time
        segment = spectrogram_2d[:, start:end]
        
        features[f'segment_{seg_idx}_mean'] = np.mean(segment)
        features[f'segment_{seg_idx}_std'] = np.std(segment)
        features[f'segment_{seg_idx}_max'] = np.max(segment)
        features[f'segment_{seg_idx}_min'] = np.min(segment)
        features[f'segment_{seg_idx}_energy'] = np.sum(segment ** 2)
    
    # 구간 간 변화율
    for seg_idx in range(n_segments - 1):
        features[f'segment_{seg_idx}_to_{seg_idx+1}_change'] = (
            features[f'segment_{seg_idx+1}_mean'] - features[f'segment_{seg_idx}_mean']
        )
    
    return features

# 모든 샘플에 대해 시간 구간 특징 추출
print("🔄 시간 구간 특징 추출 중...")
time_segment_features = []

for spec_2d in tqdm(X_2d, desc="시간 구간 추출"):
    features = extract_time_segment_features(spec_2d, n_segments=3)
    time_segment_features.append(features)

# DataFrame으로 변환
df_time_segment = pd.DataFrame(time_segment_features)

print(f"\n✅ 시간 구간 특징 추출 완료! (Shape: {df_time_segment.shape})")

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for state_idx, state_name in enumerate(state_names):
    mask = y_states == state_idx
    
    # 각 구간별 평균 비교
    segment_means = [
        df_time_segment[mask][f'segment_{i}_mean'].values for i in range(3)
    ]
    
    bp = axes[state_idx].boxplot(segment_means, labels=[f'Segment {i+1}' for i in range(3)],
                                patch_artist=True)
    
    # 박스 색상 설정
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    for patch in bp['boxes']:
        patch.set_facecolor(colors[state_idx])
        patch.set_alpha(0.7)
    
    axes[state_idx].set_title(f'{state_name} - 시간 구간별 평균', fontweight='bold')
    axes[state_idx].set_ylabel('Mean Amplitude')
    axes[state_idx].grid(True, alpha=0.3, axis='y')

plt.suptitle('⏱️ 시간 구간별 통계 분석', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()



## 8. 스펙트로그램 변화율 분석 (Gradient)



In [ ]:
# ============================================================
# 스펙트로그램 변화율 분석 (Gradient)
# ============================================================

def extract_gradient_features(spectrogram_2d):
    """
    스펙트로그램의 시간/주파수 축 변화율 분석
    
    Returns:
        dict: 변화율 특징들
    """
    features = {}
    
    # 시간 축 변화율 (시간에 따른 변화)
    time_grad = np.gradient(spectrogram_2d, axis=1)
    features['time_grad_mean'] = np.mean(time_grad)
    features['time_grad_std'] = np.std(time_grad)
    features['time_grad_max'] = np.max(time_grad)
    features['time_grad_min'] = np.min(time_grad)
    features['time_grad_abs_mean'] = np.mean(np.abs(time_grad))
    
    # 주파수 축 변화율 (주파수에 따른 변화)
    freq_grad = np.gradient(spectrogram_2d, axis=0)
    features['freq_grad_mean'] = np.mean(freq_grad)
    features['freq_grad_std'] = np.std(freq_grad)
    features['freq_grad_max'] = np.max(freq_grad)
    features['freq_grad_min'] = np.min(freq_grad)
    features['freq_grad_abs_mean'] = np.mean(np.abs(freq_grad))
    
    # 2차 변화율 (가속도)
    time_grad2 = np.gradient(time_grad, axis=1)
    freq_grad2 = np.gradient(freq_grad, axis=0)
    
    features['time_grad2_mean'] = np.mean(time_grad2)
    features['time_grad2_std'] = np.std(time_grad2)
    features['freq_grad2_mean'] = np.mean(freq_grad2)
    features['freq_grad2_std'] = np.std(freq_grad2)
    
    return features

# 모든 샘플에 대해 변화율 특징 추출
print("🔄 변화율 특징 추출 중...")
gradient_features = []

for spec_2d in tqdm(X_2d, desc="변화율 추출"):
    features = extract_gradient_features(spec_2d)
    gradient_features.append(features)

# DataFrame으로 변환
df_gradient = pd.DataFrame(gradient_features)

print(f"\n✅ 변화율 특징 추출 완료! (Shape: {df_gradient.shape})")

# 시각화
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 시간 축 변화율 평균
for state_idx, state_name in enumerate(state_names):
    mask = y_states == state_idx
    axes[0, 0].hist(df_gradient[mask]['time_grad_abs_mean'], bins=50, alpha=0.6,
                   label=state_name, color=['#FF6B6B', '#4ECDC4', '#45B7D1'][state_idx])
axes[0, 0].set_xlabel('Time Gradient (Absolute Mean)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('시간 축 변화율 분포', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. 주파수 축 변화율 평균
for state_idx, state_name in enumerate(state_names):
    mask = y_states == state_idx
    axes[0, 1].hist(df_gradient[mask]['freq_grad_abs_mean'], bins=50, alpha=0.6,
                   label=state_name, color=['#FF6B6B', '#4ECDC4', '#45B7D1'][state_idx])
axes[0, 1].set_xlabel('Frequency Gradient (Absolute Mean)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('주파수 축 변화율 분포', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. 시간 vs 주파수 변화율 산점도
for state_idx, state_name in enumerate(state_names):
    mask = y_states == state_idx
    axes[1, 0].scatter(df_gradient[mask]['time_grad_abs_mean'], 
                      df_gradient[mask]['freq_grad_abs_mean'],
                      alpha=0.5, label=state_name, s=20,
                      color=['#FF6B6B', '#4ECDC4', '#45B7D1'][state_idx])
axes[1, 0].set_xlabel('Time Gradient (Absolute Mean)')
axes[1, 0].set_ylabel('Frequency Gradient (Absolute Mean)')
axes[1, 0].set_title('시간 vs 주파수 변화율', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. 2차 변화율 비교
grad2_data = [
    df_gradient[y_states == i]['time_grad2_std'].values for i in range(3)
]
bp = axes[1, 1].boxplot(grad2_data, labels=state_names, patch_artist=True)
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1, 1].set_ylabel('Time Gradient2 (Std)')
axes[1, 1].set_title('2차 변화율 (가속도) 비교', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.suptitle('📈 스펙트로그램 변화율 분석', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# 모든 특징 결합 및 PCA 시각화
# ============================================================

# 모든 특징 결합
print("🔄 모든 특징 결합 중...")
all_features = pd.concat([
    df_stats,
    df_time_pattern,
    df_freq_pattern,
    df_time_segment,
    df_gradient
], axis=1)

print(f"✅ 특징 결합 완료! (Shape: {all_features.shape})")
print(f"   총 특징 수: {all_features.shape[1]}개")

# 데이터 정규화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(all_features)

print("\n✅ 데이터 정규화 완료!")

# PCA 수행
print("\n🔄 PCA 수행 중...")
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_3d = PCA(n_components=3, random_state=42)
X_pca_3d = pca_3d.fit_transform(X_scaled)

print(f"✅ PCA 완료!")
print(f"   2D 설명 분산: {pca_2d.explained_variance_ratio_.sum():.4f}")
print(f"   3D 설명 분산: {pca_3d.explained_variance_ratio_.sum():.4f}")



In [ ]:
# ============================================================
# PCA 2D 시각화
# ============================================================

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
markers = ['o', 's', '^']

fig, ax = plt.subplots(figsize=(12, 10))

for state_idx, (name, color, marker) in enumerate(zip(state_names, colors, markers)):
    mask = y_states == state_idx
    ax.scatter(
        X_pca_2d[mask, 0], 
        X_pca_2d[mask, 1], 
        c=color, 
        marker=marker,
        label=f'{name} (n={mask.sum()})',
        alpha=0.6,
        s=50
    )

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%})', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%})', fontsize=12)
ax.set_title('🎯 통합 특징 기반 PCA 2D 시각화', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



In [ ]:
# ============================================================
# K-means 클러스터링
# ============================================================

print("🔄 K-Means 클러스터링 수행 중...")

# K=3으로 클러스터링
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(X_scaled)

print(f"✅ K-Means 클러스터링 완료!")

# 클러스터링 성능 평가
silhouette = silhouette_score(X_scaled, cluster_labels)
ari = adjusted_rand_score(y_states, cluster_labels)
nmi = normalized_mutual_info_score(y_states, cluster_labels)

print("\n" + "=" * 60)
print("📊 클러스터링 성능 평가")
print("=" * 60)
print(f"\n1️⃣ Silhouette Score: {silhouette:.4f}")
print("   (-1 ~ 1, 높을수록 군집이 잘 분리됨)")
print(f"\n2️⃣ Adjusted Rand Index (ARI): {ari:.4f}")
print("   (0 ~ 1, 높을수록 실제 레이블과 일치)")
print(f"\n3️⃣ Normalized Mutual Information (NMI): {nmi:.4f}")
print("   (0 ~ 1, 높을수록 실제 레이블과 일치)")



In [ ]:
# ============================================================
# 클러스터링 결과 시각화
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 실제 레이블
for state_idx, (name, color, marker) in enumerate(zip(state_names, colors, markers)):
    mask = y_states == state_idx
    axes[0].scatter(
        X_pca_2d[mask, 0], X_pca_2d[mask, 1],
        c=color, marker=marker,
        label=name, alpha=0.6, s=50
    )

axes[0].set_xlabel('PC1', fontsize=12)
axes[0].set_ylabel('PC2', fontsize=12)
axes[0].set_title('🔵 실제 레이블', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 클러스터링 결과
cluster_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for cluster_id in range(3):
    mask = cluster_labels == cluster_id
    axes[1].scatter(
        X_pca_2d[mask, 0], X_pca_2d[mask, 1],
        c=cluster_colors[cluster_id], marker='o',
        label=f'Cluster {cluster_id}', alpha=0.6, s=50
    )

# 클러스터 중심점 표시
centers_pca = pca_2d.transform(kmeans.cluster_centers_)
axes[1].scatter(
    centers_pca[:, 0], centers_pca[:, 1],
    c='black', marker='X', s=200, edgecolors='white', linewidths=2,
    label='Centroids'
)

axes[1].set_xlabel('PC1', fontsize=12)
axes[1].set_ylabel('PC2', fontsize=12)
axes[1].set_title('🔵 K-Means 클러스터링 결과', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('📊 실제 레이블 vs K-Means 클러스터링', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n✅ 스펙트로그램 패턴 및 파동 분석 완료!")

